# Day 2, Session 1 -- Exercises: OpenAI API Engineering with Structured Outputs

**Engineering context:** You are the engineer building production-grade API integrations. Consultants are your users -- they need reliable structured outputs, streaming for real-time briefings, and validated data pipelines they can trust.

**How to use this notebook:**
- Each exercise builds on patterns from the demo notebook (`D2S1_demos.ipynb`)
- Follow the numbered TODO steps in order
- Hints are provided in collapsible `<details>` sections -- try first, then peek if stuck
- Exercises 1-3 are required; Optional exercises are stretch goals

In [ ]:
!pip install -q openai pydantic python-dotenv

## Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import openai
import json
import os
import time
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

## Recap from Demos

In the demos you saw how to:
- **Demo 1:** Track token usage and stream responses in real-time
- **Demo 2:** Generate guaranteed-valid JSON with `response_format={"type": "json_object"}`
- **Demo 3:** Implement the full function-calling loop (define tools, detect tool_calls, execute locally, send result back)
- **Demo 4:** Validate LLM outputs with Pydantic models
- **Demo 5:** Build reusable extraction pipelines combining JSON mode + Pydantic

Now you will build these patterns yourself.

---
## Exercise 1 (Easy): Financial Analysis Tool with Function Calling

**Reference:** Demo 3 (Function Calling)

> **Hint:** The function-calling loop has 3 phases: (1) send message with tools, (2) detect and execute tool_calls, (3) send tool result back to the model.

Build a financial analysis tool that uses OpenAI function calling. The complete flow:
1. Define a tool schema for `financial_analysis` with parameters `analysis_type` (enum) and `company` (string)
2. Implement the simulated function with hardcoded financial data
3. Handle the full function-calling loop: send question with tools, detect tool_calls, execute function, send result back

**Steps:**
1. Implement `financial_analysis()` -- the simulated data function
2. Define the `tools` list with the JSON Schema for the function
3. Implement `ask_financial_question()` with the complete request-execute-respond loop
4. Test with the provided questions

In [ ]:
# Exercise 1: Financial Analysis Tool with Function Calling

client = openai.OpenAI()

# =============================================================================
# Step 1: Simulated financial_analysis function (provided)
# =============================================================================

company_data = {
    "Meridian Health": {
        "valuation": {"ev_ebitda": 14.2, "pe_ratio": 22.5, "market_cap_bn": 18.7},
        "profitability": {"gross_margin_pct": 42.3, "ebitda_margin_pct": 18.1, "net_margin_pct": 8.9},
        "leverage": {"debt_to_equity": 1.4, "net_debt_ebitda": 3.2, "interest_coverage": 5.8},
        "liquidity": {"current_ratio": 1.8, "quick_ratio": 1.2, "cash_bn": 2.1}
    },
    "Apex Energy": {
        "valuation": {"ev_ebitda": 8.5, "pe_ratio": 15.3, "market_cap_bn": 42.1},
        "profitability": {"gross_margin_pct": 55.7, "ebitda_margin_pct": 32.4, "net_margin_pct": 14.2},
        "leverage": {"debt_to_equity": 0.8, "net_debt_ebitda": 1.9, "interest_coverage": 8.3},
        "liquidity": {"current_ratio": 2.1, "quick_ratio": 1.6, "cash_bn": 5.4}
    }
}

def financial_analysis(analysis_type, company):
    """Perform a financial analysis on a company (simulated data)."""
    if company in company_data and analysis_type in company_data[company]:
        return json.dumps({"company": company, "analysis_type": analysis_type,
                           "data": company_data[company][analysis_type]})
    return json.dumps({"error": f"No data for {company} / {analysis_type}"})


# =============================================================================
# Step 2: Define the tools list
# (Refer to Demo 3 -- the tools list structure with "type": "function")
#
# TODO: Complete the tool schema below by filling in the parameters section
# Hint: analysis_type should be an enum of ["valuation","profitability","leverage","liquidity"]
# =============================================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "financial_analysis",
            "description": "Perform financial analysis on a company. Returns metrics for the requested analysis type.",
            "parameters": {
                "type": "object",
                "properties": {
                    # TODO: Add "analysis_type" property (string, enum of 4 types)
                    # TODO: Add "company" property (string)
                },
                "required": ["analysis_type", "company"]
            }
        }
    }
]


# =============================================================================
# Step 3: Implement the function-calling loop
# (Refer to Demo 3c -- the execute-and-respond pattern)
#
# The flow is provided as comments -- uncomment and complete.
# =============================================================================

def ask_financial_question(question):
    """Send a financial analysis question and handle function calling."""
    messages = [{"role": "user", "content": question}]

    # Step 3a: First API call with tools
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    # Step 3b: Check if the model wants to call a tool
    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments)

        # TODO: Call financial_analysis with the parsed args
        # Hint: result = financial_analysis(**args)
        result = None  # <-- replace with correct call

        # Step 3c: Send the tool result back (provided)
        messages.append(msg)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

        # Step 3d: Get final response
        final = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=messages
        )
        return final.choices[0].message.content

    return msg.content


# =============================================================================
# Step 4: Test
# =============================================================================
# Uncomment to test:
# print("Test 1: Valuation")
# print(ask_financial_question("What is the valuation profile of Meridian Health?"))
# print("\n" + "="*60 + "\n")
# print("Test 2: Profitability")
# print(ask_financial_question("How profitable is Apex Energy? Give me the margin analysis."))

<details>
<summary><strong>Hint 1:</strong> financial_analysis function structure</summary>

```python
def financial_analysis(analysis_type, company):
    company_data = {
        "Meridian Health": { ... },
        "Apex Energy": { ... }
    }
    data = company_data.get(company, {}).get(analysis_type, {"error": "No data found"})
    return json.dumps({"company": company, "analysis_type": analysis_type, **data})
```
</details>

<details>
<summary><strong>Hint 2:</strong> Tool schema structure</summary>

```python
tools = [
    {
        "type": "function",
        "function": {
            "name": "financial_analysis",
            "description": "Get financial analysis data for a company",
            "parameters": {
                "type": "object",
                "properties": {
                    "analysis_type": {
                        "type": "string",
                        "enum": ["valuation", "profitability", "leverage", "liquidity"]
                    },
                    "company": {"type": "string"}
                },
                "required": ["analysis_type", "company"]
            }
        }
    }
]
```
</details>

<details>
<summary><strong>Hint 3:</strong> Function-calling loop</summary>

```python
def ask_financial_question(question):
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto"
    )
    message = response.choices[0].message
    
    if message.tool_calls:
        tool_call = message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        result = financial_analysis(**args)
        
        follow_up = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=[
                {"role": "user", "content": question},
                message,
                {"role": "tool", "tool_call_id": tool_call.id, "content": result}
            ],
            tools=tools
        )
        return follow_up.choices[0].message.content
    return message.content
```
</details>

### Expected Output

Your output will vary in wording since the LLM generates natural language, but the data values should match:

```
Test 1: Valuation
The valuation profile of Meridian Health includes:
- EV/EBITDA: 14.2
- P/E Ratio: 22.5
- Market Capitalization: $18.7 billion

Test 2: Profitability
Apex Energy has the following profitability margins:
- Gross Margin: 55.7%
- EBITDA Margin: 32.4%
- Net Margin: 14.2%
```

---
## Exercise 2 (Easy): Streaming Response Collector with Token Counter

**Reference:** Demo 1 (Streaming + Token Tracking)

> **Hint:** Streaming uses a for-loop over the response chunks. Each chunk has a `.choices[0].delta.content` field.

Build a function that streams a response, counts tokens in real-time, and estimates cost. This is a common utility in production systems where you need to monitor spend per request.

**Steps:**
1. Create a function `stream_with_metrics()` that takes a prompt
2. Stream the response, collecting text and counting chunks
3. After streaming completes, estimate token count and cost
4. Return a dictionary with the response text and metrics

In [ ]:
# Exercise 2: Streaming Response Collector with Token Counter

client = openai.OpenAI()

# Pricing per 1M tokens (approximate for gpt-4o-mini)
INPUT_COST_PER_1M = 0.15   # $0.15 per 1M input tokens
OUTPUT_COST_PER_1M = 0.60  # $0.60 per 1M output tokens

def stream_with_metrics(prompt, system_message="You are a helpful McKinsey consultant assistant.", max_tokens=200):
    """
    Stream a response from the API while collecting metrics.
    
    Returns:
        dict with keys: text, chunk_count, estimated_output_tokens, elapsed_seconds, estimated_cost_usd
    """
    # Step 1: Record start time (provided)
    start_time = time.time()
    collected_text = ""
    chunk_count = 0

    # Step 2: Start the streaming API call (provided)
    stream = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        stream=True
    )

    # TODO Step 3: Iterate over the stream and collect text + count chunks
    # Hint: for chunk in stream:
    #     content = chunk.choices[0].delta.content
    #     if content:
    #         collected_text += content
    #         chunk_count += 1
    #         print(content, end="", flush=True)


    # Step 4: Calculate metrics (provided)
    elapsed = time.time() - start_time
    est_output_tokens = len(collected_text) / 4
    est_input_tokens = len(prompt + system_message) / 4
    est_cost = (est_input_tokens * INPUT_COST_PER_1M / 1_000_000) + \
               (est_output_tokens * OUTPUT_COST_PER_1M / 1_000_000)

    # Step 5: Return metrics (provided)
    return {
        "text": collected_text,
        "chunk_count": chunk_count,
        "estimated_output_tokens": est_output_tokens,
        "elapsed_seconds": elapsed,
        "estimated_cost_usd": est_cost
    }


# Test the function (uncomment when ready)
# print("=" * 60)
# print("Streaming with metrics:")
# print("=" * 60)
# result = stream_with_metrics(
#     "What are the 3 most important due diligence items when evaluating a healthcare acquisition target?"
# )
# print("\n")
# print(f"  Chunks received     : {result['chunk_count']}")
# print(f"  Est. output tokens  : {result['estimated_output_tokens']:.0f}")
# print(f"  Elapsed time        : {result['elapsed_seconds']:.2f}s")
# print(f"  Estimated cost      : ${result['estimated_cost_usd']:.6f}")

<details>
<summary><strong>Hint 1:</strong> Setting up the stream</summary>

```python
start_time = time.time()
stream = client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ],
    max_tokens=max_tokens,
    stream=True
)
```
</details>

<details>
<summary><strong>Hint 2:</strong> Collecting chunks</summary>

```python
collected_text = ""
chunk_count = 0
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        chunk_count += 1
        print(token, end="", flush=True)
```
</details>

<details>
<summary><strong>Hint 3:</strong> Calculating metrics</summary>

```python
elapsed = time.time() - start_time
est_output_tokens = len(collected_text) / 4
est_input_tokens = len(prompt + system_message) / 4
cost = (est_input_tokens * INPUT_COST_PER_1M / 1_000_000) + (est_output_tokens * OUTPUT_COST_PER_1M / 1_000_000)
return {
    "text": collected_text,
    "chunk_count": chunk_count,
    "estimated_output_tokens": est_output_tokens,
    "elapsed_seconds": elapsed,
    "estimated_cost_usd": cost
}
```
</details>

---
## Exercise 3 (Easy): Multi-Tool Financial Advisor

**Reference:** Demo 3 (Function Calling)

> **Hint:** The function-calling loop has 3 phases: (1) send message with tools, (2) detect and execute tool_calls, (3) send tool result back to the model.

Build a system with 3 different tools that the model can choose from. The model should select the appropriate tool based on the user's question.

**Tools to implement:**
1. `valuation_analysis` -- Returns valuation metrics (EV/EBITDA, P/E, market cap)
2. `profitability_analysis` -- Returns margin metrics (gross, EBITDA, net)
3. `risk_assessment` -- Returns risk metrics (beta, volatility, debt rating)

**Steps:**
1. Implement 3 simulated functions
2. Define 3 tool schemas
3. Build `ask_advisor()` that handles any tool the model selects
4. Test with questions targeting different tools

In [ ]:
# Exercise 3: Multi-Tool Financial Advisor

client = openai.OpenAI()

# =============================================================================
# Step 1: Three simulated analysis functions (provided)
# =============================================================================

def valuation_analysis(company):
    """Get valuation metrics for a company."""
    data = {
        "Meridian Health": {"ev_ebitda": 14.2, "pe_ratio": 22.5, "market_cap_bn": 18.7, "ev_revenue": 3.8},
        "Apex Energy": {"ev_ebitda": 8.5, "pe_ratio": 15.3, "market_cap_bn": 42.1, "ev_revenue": 2.1},
        "TechCorp Global": {"ev_ebitda": 28.3, "pe_ratio": 45.2, "market_cap_bn": 125.0, "ev_revenue": 12.5}
    }
    return json.dumps({"company": company, "metrics": data.get(company, {})})

def profitability_analysis(company):
    """Get profitability metrics for a company."""
    data = {
        "Meridian Health": {"gross_margin_pct": 42.3, "ebitda_margin_pct": 18.1, "net_margin_pct": 8.9, "roic_pct": 12.4},
        "Apex Energy": {"gross_margin_pct": 55.7, "ebitda_margin_pct": 32.4, "net_margin_pct": 14.2, "roic_pct": 18.7},
        "TechCorp Global": {"gross_margin_pct": 72.1, "ebitda_margin_pct": 38.5, "net_margin_pct": 22.3, "roic_pct": 25.1}
    }
    return json.dumps({"company": company, "metrics": data.get(company, {})})

def risk_assessment(company):
    """Get risk assessment metrics for a company."""
    data = {
        "Meridian Health": {"beta": 0.85, "volatility_pct": 18.2, "debt_rating": "BBB+", "default_probability_pct": 0.4},
        "Apex Energy": {"beta": 1.35, "volatility_pct": 28.7, "debt_rating": "BBB", "default_probability_pct": 0.8},
        "TechCorp Global": {"beta": 1.52, "volatility_pct": 32.1, "debt_rating": "A-", "default_probability_pct": 0.2}
    }
    return json.dumps({"company": company, "metrics": data.get(company, {})})


# =============================================================================
# Step 2: TODO -- Define the tools list with 3 tool schemas
# Each tool has a single parameter: company (string)
# Hint: Same format as Exercise 1 but with 3 entries. Each tool name
# matches the function name above. No enum needed -- just "company" (string).
# =============================================================================

tools = [
    # TODO: Add 3 tool schemas (one per function above)
]


# =============================================================================
# Step 3: Function dispatcher (provided)
# =============================================================================

function_map = {
    "valuation_analysis": valuation_analysis,
    "profitability_analysis": profitability_analysis,
    "risk_assessment": risk_assessment,
}


# =============================================================================
# Step 4: TODO -- Implement ask_advisor()
# Same pattern as Exercise 1 ask_financial_question, but use function_map
# to dispatch: function_map[tool_call.function.name](**args)
# =============================================================================

def ask_advisor(question):
    """Ask the financial advisor a question. It will choose the right tool."""
    messages = [{"role": "user", "content": question}]

    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    msg = response.choices[0].message

    if msg.tool_calls:
        tool_call = msg.tool_calls[0]
        args = json.loads(tool_call.function.arguments)

        # TODO: Call the correct function using function_map
        # Hint: result = function_map[tool_call.function.name](**args)
        result = None  # <-- replace

        messages.append(msg)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

        final = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=messages
        )
        return final.choices[0].message.content

    return msg.content


# Step 5: Test (uncomment when ready)
# print("Test 1 (should use valuation_analysis):")
# print(ask_advisor("What is TechCorp Global's valuation? I need EV/EBITDA and market cap."))
# print("\n" + "="*60 + "\n")
# print("Test 2 (should use profitability_analysis):")
# print(ask_advisor("How profitable is Meridian Health? Show me their margins."))
# print("\n" + "="*60 + "\n")
# print("Test 3 (should use risk_assessment):")
# print(ask_advisor("What is the risk profile of Apex Energy? I need beta and debt rating."))

<details>
<summary><strong>Hint 1:</strong> Function implementation pattern</summary>

```python
def valuation_analysis(company):
    data = { ... }  # already provided above
    metrics = data.get(company, {"error": f"No data for {company}"})
    return json.dumps({"company": company, "analysis": "valuation", **metrics})
```
</details>

<details>
<summary><strong>Hint 2:</strong> Tool schema for one function</summary>

```python
{
    "type": "function",
    "function": {
        "name": "valuation_analysis",
        "description": "Get valuation metrics (EV/EBITDA, P/E ratio, market cap) for a company",
        "parameters": {
            "type": "object",
            "properties": {
                "company": {"type": "string", "description": "Company name"}
            },
            "required": ["company"]
        }
    }
}
```
</details>

<details>
<summary><strong>Hint 3:</strong> Dispatching the correct function</summary>

```python
function_map = {
    "valuation_analysis": valuation_analysis,
    "profitability_analysis": profitability_analysis,
    "risk_assessment": risk_assessment
}

# In ask_advisor, after detecting tool_calls:
tool_call = message.tool_calls[0]
func_name = tool_call.function.name
args = json.loads(tool_call.function.arguments)
result = function_map[func_name](**args)
```
</details>

---
---
# Optional Exercises


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
These exercises are at an intermediate level and combine multiple patterns. Attempt them if you finish the required exercises early.

---
## Optional Exercise 1 (Intermediate): Validated Data Pipeline

**Reference:** Demo 4 + Demo 5 (Pydantic Validation + Extraction Pipeline)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Build an extraction pipeline that:
1. Takes unstructured meeting notes as input
2. Extracts structured client engagement data using JSON mode
3. Validates the output with a Pydantic model
4. Handles validation errors gracefully (logs the error, returns partial data)

**Scenario:** You receive messy meeting notes from consultants and need to extract structured engagement metadata for the CRM system.

In [ ]:
# Optional Exercise 1: Validated Data Pipeline

client = openai.OpenAI()

# =============================================================================
# TODO Step 1: Define a Pydantic model for ClientEngagement
# Fields:
#   - client_name: str (required)
#   - industry: str (required)
#   - engagement_type: str (required)
#   - budget_usd: Optional[float] (default None)
#   - start_date: Optional[str] (default None)
#   - team_lead: Optional[str] (default None)
#   - key_objectives: List[str] (default empty list)
#   - risks: List[str] (default empty list)
# =============================================================================

class ClientEngagement(BaseModel):
    # YOUR CODE HERE
    pass


# =============================================================================
# TODO Step 2: Build the extraction function
# - Use JSON mode to get structured output
# - Validate with Pydantic
# - If validation fails, catch the error and return a dict with:
#   {"success": False, "error": str(e), "raw_json": raw_response}
# - If validation passes, return:
#   {"success": True, "data": validated_model, "raw_json": raw_response}
# =============================================================================

def extract_engagement_data(meeting_notes: str) -> dict:
    """Extract structured engagement data from meeting notes."""
    # YOUR CODE HERE
    pass


# =============================================================================
# Step 3: Test with sample meeting notes
# =============================================================================

sample_notes = """
Meeting Notes - April 15, 2025
Client: NovaTech Solutions (Technology/SaaS sector)

We kicked off the digital transformation engagement today. Sarah mentioned the
budget is around $2.5M for Phase 1. John Park from our Strategy practice will
lead the team. We're targeting a June 1 start date.

Key objectives discussed:
- Migrate legacy systems to cloud-native architecture
- Implement AI-powered customer service chatbot
- Redesign the data analytics platform

Risks flagged by the client:
- Tight timeline (6 months for Phase 1)
- Key technical staff leaving in Q3
- Integration complexity with legacy ERP
"""

result = extract_engagement_data(sample_notes)
if result["success"]:
    engagement = result["data"]
    print("Extraction SUCCESSFUL")
    print(f"  Client: {engagement.client_name}")
    print(f"  Industry: {engagement.industry}")
    print(f"  Type: {engagement.engagement_type}")
    print(f"  Budget: ${engagement.budget_usd:,.0f}" if engagement.budget_usd else "  Budget: N/A")
    print(f"  Start: {engagement.start_date or 'N/A'}")
    print(f"  Lead: {engagement.team_lead or 'N/A'}")
    print(f"  Objectives: {engagement.key_objectives}")
    print(f"  Risks: {engagement.risks}")
else:
    print(f"Extraction FAILED: {result['error']}")
    print(f"Raw JSON: {result['raw_json']}")

<details>
<summary><strong>Hint:</strong> Extraction function structure</summary>

```python
def extract_engagement_data(meeting_notes: str) -> dict:
    response = client.chat.completions.create(
        model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
        messages=[
            {"role": "system", "content": "Extract client engagement data from meeting notes. Return JSON with: client_name, industry, engagement_type, budget_usd (number or null), start_date (string or null), team_lead (string or null), key_objectives (list of strings), risks (list of strings)."},
            {"role": "user", "content": meeting_notes}
        ],
        response_format={"type": "json_object"},
        max_tokens=500
    )
    raw = response.choices[0].message.content
    try:
        data = ClientEngagement.model_validate_json(raw)
        return {"success": True, "data": data, "raw_json": raw}
    except ValidationError as e:
        return {"success": False, "error": str(e), "raw_json": raw}
```
</details>

---
## Optional Exercise 2 (Intermediate): Auto-Retry with Schema Correction

**Reference:** Demo 4 (Pydantic Validation)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Build a function that:
1. Calls the LLM for JSON output
2. Validates with Pydantic
3. If validation fails, sends the validation error message back to the LLM and asks it to fix the output
4. Retries up to N times before giving up

This pattern is extremely useful in production -- LLMs occasionally produce slightly malformed outputs, and an auto-retry with error feedback usually fixes it on the second attempt.

In [ ]:
# Optional Exercise 2: Auto-Retry with Schema Correction

client = openai.OpenAI()

# Target schema -- deliberately strict to test retry logic
class ProjectProposal(BaseModel):
    title: str = Field(min_length=5, description="Project title (at least 5 characters)")
    client: str = Field(description="Client organization name")
    duration_weeks: int = Field(ge=1, le=52, description="Duration in weeks (1-52)")
    team_size: int = Field(ge=1, le=50, description="Number of team members (1-50)")
    estimated_value_usd: float = Field(gt=0, description="Estimated project value in USD (must be positive)")
    methodology: str = Field(description="Primary methodology: one of 'agile', 'waterfall', 'hybrid'")
    key_deliverables: List[str] = Field(min_length=2, description="At least 2 key deliverables")


# =============================================================================
# TODO: Implement validated_llm_call with auto-retry
#
# Parameters:
#   - prompt: the user prompt
#   - model_class: the Pydantic model class to validate against
#   - max_retries: maximum number of retry attempts (default 3)
#
# Logic:
#   1. Make initial LLM call with JSON mode
#   2. Try to validate with model_class.model_validate_json(raw)
#   3. If validation passes, return {"success": True, "data": validated, "attempts": attempt_count}
#   4. If validation fails:
#      a. Build a new prompt that includes the original request, the invalid JSON,
#         and the validation error message
#      b. Ask the LLM to fix it
#      c. Repeat up to max_retries times
#   5. If all retries fail, return {"success": False, "error": last_error, "attempts": max_retries}
# =============================================================================

def validated_llm_call(prompt, model_class, max_retries=3):
    """Call LLM with auto-retry on validation failure."""
    # YOUR CODE HERE
    pass


# Test
result = validated_llm_call(
    "Create a project proposal for a 12-week digital transformation engagement with GlobalBank Corp. Team of 8. Value around $1.5M. Use agile methodology.",
    ProjectProposal
)

if result["success"]:
    proposal = result["data"]
    print(f"Validated in {result['attempts']} attempt(s)")
    print(f"  Title: {proposal.title}")
    print(f"  Client: {proposal.client}")
    print(f"  Duration: {proposal.duration_weeks} weeks")
    print(f"  Team: {proposal.team_size} members")
    print(f"  Value: ${proposal.estimated_value_usd:,.0f}")
    print(f"  Methodology: {proposal.methodology}")
    print(f"  Deliverables: {proposal.key_deliverables}")
else:
    print(f"Failed after {result['attempts']} attempts")
    print(f"Last error: {result['error']}")

<details>
<summary><strong>Hint:</strong> Auto-retry loop structure</summary>

```python
def validated_llm_call(prompt, model_class, max_retries=3):
    system_msg = f"Output JSON matching this schema: {json.dumps(model_class.model_json_schema())}"
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt}
    ]
    
    for attempt in range(1, max_retries + 1):
        response = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=messages,
            response_format={"type": "json_object"},
            max_tokens=500
        )
        raw = response.choices[0].message.content
        
        try:
            data = model_class.model_validate_json(raw)
            return {"success": True, "data": data, "attempts": attempt}
        except ValidationError as e:
            error_msg = str(e)
            # Add the failed attempt and error to messages for retry
            messages.append({"role": "assistant", "content": raw})
            messages.append({"role": "user", "content": f"That JSON failed validation: {error_msg}\nPlease fix the JSON to match the schema exactly."})
    
    return {"success": False, "error": error_msg, "attempts": max_retries}
```
</details>

---
## Optional Exercise 3 (Intermediate): Multi-Step Tool Orchestrator

**Reference:** Demo 3 + Demo 4 (Function Calling + Pydantic)


> **Hint:** Review the referenced demos for the exact patterns to follow. The optional exercises combine multiple demo patterns.
Build a system where the LLM can call multiple tools **in sequence** to answer a complex question. For example: "Compare the valuation and profitability of Meridian Health vs Apex Energy" requires 4 tool calls (2 companies x 2 analysis types).

**Key challenge:** You must handle multiple rounds of tool calls in a loop until the model produces a final text answer (no more tool_calls).

**Steps:**
1. Reuse the 3 tools from Exercise 3
2. Build a loop that keeps sending tool results back until the model stops requesting tools
3. Validate the final comparison output with Pydantic

In [ ]:
# Optional Exercise 3: Multi-Step Tool Orchestrator

client = openai.OpenAI()

# =============================================================================
# Reuse the tools and functions from Exercise 3
# (Copy your implementations here or re-define them)
# =============================================================================

def valuation_analysis(company):
    data = {
        "Meridian Health": {"ev_ebitda": 14.2, "pe_ratio": 22.5, "market_cap_bn": 18.7, "ev_revenue": 3.8},
        "Apex Energy": {"ev_ebitda": 8.5, "pe_ratio": 15.3, "market_cap_bn": 42.1, "ev_revenue": 2.1},
        "TechCorp Global": {"ev_ebitda": 28.3, "pe_ratio": 45.2, "market_cap_bn": 125.0, "ev_revenue": 12.5}
    }
    metrics = data.get(company, {"error": f"No data for {company}"})
    return json.dumps({"company": company, "analysis": "valuation", **metrics})

def profitability_analysis(company):
    data = {
        "Meridian Health": {"gross_margin_pct": 42.3, "ebitda_margin_pct": 18.1, "net_margin_pct": 8.9, "roic_pct": 12.4},
        "Apex Energy": {"gross_margin_pct": 55.7, "ebitda_margin_pct": 32.4, "net_margin_pct": 14.2, "roic_pct": 18.7},
        "TechCorp Global": {"gross_margin_pct": 72.1, "ebitda_margin_pct": 38.5, "net_margin_pct": 22.3, "roic_pct": 25.1}
    }
    metrics = data.get(company, {"error": f"No data for {company}"})
    return json.dumps({"company": company, "analysis": "profitability", **metrics})

def risk_assessment(company):
    data = {
        "Meridian Health": {"beta": 0.85, "volatility_pct": 18.2, "debt_rating": "BBB+", "default_probability_pct": 0.4},
        "Apex Energy": {"beta": 1.35, "volatility_pct": 28.7, "debt_rating": "BBB", "default_probability_pct": 0.8},
        "TechCorp Global": {"beta": 1.52, "volatility_pct": 32.1, "debt_rating": "A-", "default_probability_pct": 0.2}
    }
    metrics = data.get(company, {"error": f"No data for {company}"})
    return json.dumps({"company": company, "analysis": "risk", **metrics})

function_map = {
    "valuation_analysis": valuation_analysis,
    "profitability_analysis": profitability_analysis,
    "risk_assessment": risk_assessment
}

tools = [
    {
        "type": "function",
        "function": {
            "name": "valuation_analysis",
            "description": "Get valuation metrics (EV/EBITDA, P/E ratio, market cap) for a company",
            "parameters": {
                "type": "object",
                "properties": {"company": {"type": "string", "description": "Company name"}},
                "required": ["company"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "profitability_analysis",
            "description": "Get profitability metrics (gross margin, EBITDA margin, net margin, ROIC) for a company",
            "parameters": {
                "type": "object",
                "properties": {"company": {"type": "string", "description": "Company name"}},
                "required": ["company"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "risk_assessment",
            "description": "Get risk metrics (beta, volatility, debt rating, default probability) for a company",
            "parameters": {
                "type": "object",
                "properties": {"company": {"type": "string", "description": "Company name"}},
                "required": ["company"]
            }
        }
    }
]


# =============================================================================
# TODO: Implement multi_step_query()
#
# This function must handle MULTIPLE ROUNDS of tool calls.
# The loop continues until the model returns a message without tool_calls.
#
# Algorithm:
#   1. Start with messages = [{"role": "user", "content": question}]
#   2. Call the API with tools
#   3. If response has tool_calls:
#      a. Add the assistant message to the messages list
#      b. For EACH tool_call in message.tool_calls:
#         - Execute the function using function_map
#         - Append {"role": "tool", "tool_call_id": ..., "content": result}
#      c. Go back to step 2
#   4. If response has NO tool_calls:
#      - Return the final message content
#   5. Safety: limit to max_rounds iterations to avoid infinite loops
# =============================================================================

def multi_step_query(question, max_rounds=5):
    """Handle multi-round tool calling until the model produces a final answer."""
    # YOUR CODE HERE
    pass


# =============================================================================
# Test with a complex question requiring multiple tool calls
# =============================================================================
print("Complex Query: Comparing two companies across multiple dimensions")
print("=" * 70)
result = multi_step_query(
    "Compare the valuation and profitability of Meridian Health vs Apex Energy. "
    "Which company offers better value relative to its profitability?"
)
print(result)
print("\n" + "=" * 70 + "\n")

print("Complex Query: Full risk-value assessment")
print("=" * 70)
result2 = multi_step_query(
    "I'm considering investing in TechCorp Global. Give me a complete picture: "
    "valuation, profitability, and risk assessment. Is it overvalued given its risk?"
)
print(result2)

<details>
<summary><strong>Hint:</strong> Multi-round loop structure</summary>

```python
def multi_step_query(question, max_rounds=5):
    messages = [{"role": "user", "content": question}]
    
    for round_num in range(max_rounds):
        response = client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        message = response.choices[0].message
        
        if not message.tool_calls:
            # No more tool calls -- we have our final answer
            return message.content
        
        # Add assistant message with tool_calls to history
        messages.append(message)
        
        # Execute ALL tool calls in this round
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            result = function_map[func_name](**args)
            print(f"  [Round {round_num+1}] Called {func_name}({args}) -> {result[:50]}...")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })
    
    return "Max rounds reached without final answer."
```
</details>

---
## Summary

In these exercises you practiced:

| Exercise | Pattern | Key Skill |
|----------|---------|----------|
| 1 | Function Calling | Define tool schemas, handle the request-execute-respond loop |
| 2 | Streaming + Metrics | Real-time token collection, cost estimation |
| 3 | Multi-Tool Selection | Model chooses from multiple tools, function dispatch |
| Opt 1 | Validated Pipeline | JSON mode + Pydantic for unstructured data extraction |
| Opt 2 | Auto-Retry | Error feedback loop for self-correcting LLM outputs |
| Opt 3 | Multi-Step Orchestration | Multiple rounds of tool calls for complex queries |

**Key Engineering Principles:**
- Always validate LLM outputs before trusting them
- Use function calling to keep the model sandboxed
- Stream responses for better UX in production
- Track tokens and costs for budget management
- Build retry logic for production resilience